# Colab Bootstrap for 10k JWST Pilot


This notebook bootstraps the current pilot workflow for the JWST SSL ViT project on Google Colab.


The corrected pilot matrix is `3 SSL methods x 3 ViT sizes` with the framework fixed to `timm` for the first pass:


- MAE: tiny, small, base
- DINO: tiny, small, base
- MAE->DINO: tiny, small, base


The `pilot_split` CSVs are not used for SSL pretraining itself. The 9 pretraining runs use the full unlabeled 10k corpus in `resized_10k_files`. The split files are for later evaluation, labeling, or supervised probe work.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/vision-transformer')
DATA_ROOT = DRIVE_PROJECT_ROOT / 'data' / 'JWST'
CATALOG_DIR = DATA_ROOT / 'resized_10k_files'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'output' / 'experiments'
SPLIT_ROOT = DATA_ROOT / 'pilot_split_colab'

print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('CATALOG_DIR =', CATALOG_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
%%bash
set -e
cd /content
rm -rf JWST-vision-transformer
git clone https://github.com/nntran15/JWST-vision-transformer.git
cd /content/JWST-vision-transformer
pip install -r requirements/pytorch.txt

In [ ]:
import os
import sys

REPO_ROOT = Path('/content/JWST-vision-transformer')
MANIFEST_CSV = DATA_ROOT / 'resized_10k_files_manifest.csv'
assert REPO_ROOT.exists(), 'Repo clone failed.'
assert CATALOG_DIR.exists(), f'Missing resized corpus directory: {CATALOG_DIR}'

os.chdir(REPO_ROOT)
print('Python executable:', sys.executable)
print('Repo root:', REPO_ROOT)
print('Catalog file count:', len(list(CATALOG_DIR.glob('*.fits'))))
print('Manifest found:', MANIFEST_CSV.exists())

## Optional: Make the Evaluation Splits Portable on Colab

If `resized_10k_files_manifest.csv` is present on Drive, the next cell regenerates the filter-aware split on Drive and adds a `runtime_resized_path` column built from `resized_relpath`.

If you only uploaded the 10k FITS files and not the manifest, you can skip this section. The training, embedding extraction, clustering, labeling, and classifier steps below will still run.

In [ ]:
import csv
import subprocess

SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

if not MANIFEST_CSV.exists():
    print('Skipping split generation because the manifest is not on Drive.')
else:
    subprocess.run([
        sys.executable,
        'scripts/1-data_preparation/create_filter_aware_pilot_split.py',
        '--manifest-csv', str(MANIFEST_CSV),
        '--output-dir', str(SPLIT_ROOT),
        '--overwrite',
    ], cwd=REPO_ROOT, check=True)

    for split_name in ('train', 'val'):
        input_csv = SPLIT_ROOT / f'{split_name}.csv'
        output_csv = SPLIT_ROOT / f'{split_name}_colab.csv'
        with input_csv.open(newline='') as src_handle, output_csv.open('w', newline='') as dst_handle:
            reader = csv.DictReader(src_handle)
            fieldnames = list(reader.fieldnames or [])
            if 'runtime_resized_path' not in fieldnames:
                fieldnames.append('runtime_resized_path')
            writer = csv.DictWriter(dst_handle, fieldnames=fieldnames)
            writer.writeheader()
            for row in reader:
                relpath = row.get('resized_relpath', '')
                row['runtime_resized_path'] = str(DATA_ROOT / relpath) if relpath else ''
                writer.writerow(row)

    print('Portable split files written to:', SPLIT_ROOT)
    print('Train split:', SPLIT_ROOT / 'train_colab.csv')
    print('Val split:', SPLIT_ROOT / 'val_colab.csv')

## Training Helper


The helper below launches the current pilot configuration: `framework=timm`, `max_samples=10000`, and one output directory per method/size pair.

In [ ]:
import subprocess
import sys

CONFIG_MAP = {
    'mae': 'configs/mae.yaml',
    'dino': 'configs/dino.yaml',
    'mae_dino': 'configs/mae_dino.yaml',
}

def run_train(method: str, vit_size: str) -> None:
    output_dir = OUTPUT_ROOT / f'pilot_{method}_timm_{vit_size}'
    checkpoint_path = output_dir / 'checkpoints' / 'checkpoint_latest.pt'
    cmd = [
        sys.executable,
        'scripts/train.py',
        '--config', CONFIG_MAP[method],
        '--framework', 'timm',
        '--vit_size', vit_size,
        '--catalog_dir', str(CATALOG_DIR),
        '--max_samples', '10000',
        '--output_dir', str(output_dir),
    ]
    if checkpoint_path.exists():
        cmd.extend(['--resume', str(checkpoint_path)])
        print('Resuming from:', checkpoint_path)
    print('Running:', ' '.join(cmd))
    
    # Use Popen to stream stdout and stderr in real-time
    process = subprocess.Popen(
        cmd, 
        cwd=REPO_ROOT, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True
    )
    
    for line in process.stdout:
        print(line, end='')
        
    process.wait()
    if process.returncode != 0:
        print(f"\n--- SUBPROCESS FAILED WITH EXIT CODE {process.returncode} ---")
        raise subprocess.CalledProcessError(process.returncode, cmd)

In [ ]:
import tarfile
from pathlib import Path

LOCAL_CATALOG_DIR = Path('/content/resized_10k_files')

# Look for a tar file in the JWST data folder
tar_files = list(DATA_ROOT.glob('*.tar*'))

if not LOCAL_CATALOG_DIR.exists() and tar_files:
    target_tar = tar_files[0]
    print(f"Found tar file: {target_tar}")
    print(f"Extracting to {LOCAL_CATALOG_DIR.parent}...")
    
    with tarfile.open(target_tar, 'r:*') as tar_ref:
        tar_ref.extractall('/content/')
        
    print("Extraction complete!")
elif not tar_files and not LOCAL_CATALOG_DIR.exists():
    print(f"Warning: No .tar or .tar.gz files found in {DATA_ROOT}!")
else:
    print(f"Local directory {LOCAL_CATALOG_DIR} already exists. Skipping extraction.")

# Update CATALOG_DIR to point to the local runtime path
CATALOG_DIR = LOCAL_CATALOG_DIR
print(f"\nCATALOG_DIR is now set to: {CATALOG_DIR}")

In [ ]:
for vit_size in ('tiny', 'small', 'base'):
    run_train('mae', vit_size)

In [ ]:
for vit_size in ('tiny', 'small', 'base'):
    run_train('dino', vit_size)

In [ ]:
for vit_size in ('tiny', 'small', 'base'):
    run_train('mae_dino', vit_size)

## After Training


1. Use the saved experiment directories in `OUTPUT_ROOT` for embedding extraction and evaluation.
2. Rank the 9 pilot runs primarily by embedding quality and linear-probe performance.
3. Only after you pick the winning method/size pair should you widen the study to alternative frameworks or larger-scale training.

## Post-Training on Colab


Embedding extraction, clustering, UMAP, and classifier training are reasonable to keep on Colab because they reuse the same GPU-backed environment and Drive-mounted outputs.


Cluster labeling itself is not compute-heavy, but it is part of the same workflow. With the notebook-friendly display backend, you can now run labeling here as well.

In [ ]:
EXPERIMENTS = [
    ('mae', 'tiny'),
    ('mae', 'small'),
    ('mae', 'base'),
    ('dino', 'tiny'),
    ('dino', 'small'),
    ('dino', 'base'),
    ('mae_dino', 'tiny'),
    ('mae_dino', 'small'),
    ('mae_dino', 'base'),
]

def get_method_for_extraction(method: str) -> str:
    return 'dino' if method == 'mae_dino' else method

def experiment_dir(method: str, vit_size: str) -> Path:
    return OUTPUT_ROOT / f'pilot_{method}_timm_{vit_size}'

def checkpoint_path(method: str, vit_size: str) -> Path:
    ckpt_dir = experiment_dir(method, vit_size) / 'checkpoints'
    if method == 'mae_dino':
        ckpt_dir = ckpt_dir / 'dino_stage2'
    
    best_pt = ckpt_dir / 'checkpoint_best.pt'
    if best_pt.exists():
        return best_pt
    
    # Fallback to checkpoint_latest.pt if best doesn't exist
    latest_pt = ckpt_dir / 'checkpoint_latest.pt'
    if latest_pt.exists():
        return latest_pt
        
    return best_pt # default return if neither exists

def embeddings_path(method: str, vit_size: str) -> Path:
    return experiment_dir(method, vit_size) / 'embeddings.h5'

def eval_dir(method: str, vit_size: str) -> Path:
    return experiment_dir(method, vit_size) / 'eval'

def run_extract_embeddings(method: str, vit_size: str, max_samples: int = 10000) -> None:
    method_for_extract = get_method_for_extraction(method)
    output_path = embeddings_path(method, vit_size)
    cmd = [
        sys.executable,
        'scripts/extract_embeddings.py',
        '--checkpoint', str(checkpoint_path(method, vit_size)),
        '--framework', 'timm',
        '--method', method_for_extract,
        '--vit_size', vit_size,
        '--catalog_dir', str(CATALOG_DIR),
        '--max_samples', str(max_samples),
        '--output', str(output_path),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

def run_cluster_eval(method: str, vit_size: str, n_clusters: int = 10) -> None:
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--embeddings', str(embeddings_path(method, vit_size)),
        '--output_dir', str(eval_dir(method, vit_size)),
        '--sweep_k',
        '--n_clusters', str(n_clusters),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

def run_labeling(method: str, vit_size: str, reps_per_cluster: int = 20) -> None:
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--embeddings', str(embeddings_path(method, vit_size)),
        '--output_dir', str(eval_dir(method, vit_size)),
        '--label',
        '--reps_per_cluster', str(reps_per_cluster),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

def run_classification(
    method: str,
    vit_size: str,
    fine_tune: bool = False,
    labels_csv: Path | None = None,
    linear_probe: bool = True,
) -> None:
    flags = []
    if linear_probe:
        flags.append('--linear_probe')
    if fine_tune:
        flags.append('--fine_tune')
    effective_labels_csv = labels_csv or (eval_dir(method, vit_size) / 'labels.csv')
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--classify',
        '--labels_csv', str(effective_labels_csv),
        '--checkpoint', str(checkpoint_path(method, vit_size)),
        '--framework', 'timm',
        '--method', get_method_for_extraction(method),
        '--vit_size', vit_size,
        '--output_dir', str(eval_dir(method, vit_size)),
    ] + flags
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

In [ ]:
def run_extract_embeddings(method: str, vit_size: str, max_samples: int = 10000) -> None:
    method_for_extract = get_method_for_extraction(method)
    output_path = embeddings_path(method, vit_size)
    cmd = [
        sys.executable,
        'scripts/extract_embeddings.py',
        '--checkpoint', str(checkpoint_path(method, vit_size)),
        '--framework', 'timm',
        '--method', method_for_extract,
        '--vit_size', vit_size,
        '--catalog_dir', str(CATALOG_DIR),
        '--max_samples', str(max_samples),
        '--output', str(output_path),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

def run_cluster_eval(method: str, vit_size: str, n_clusters: int = 10) -> None:
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--embeddings', str(embeddings_path(method, vit_size)),
        '--output_dir', str(eval_dir(method, vit_size)),
        '--sweep_k',
        '--n_clusters', str(n_clusters),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

def run_labeling(method: str, vit_size: str, reps_per_cluster: int = 20) -> None:
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--embeddings', str(embeddings_path(method, vit_size)),
        '--output_dir', str(eval_dir(method, vit_size)),
        '--label',
        '--reps_per_cluster', str(reps_per_cluster),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

def run_classification(
    method: str,
    vit_size: str,
    fine_tune: bool = False,
    labels_csv: Path | None = None,
) -> None:
    flags = ['--linear_probe']
    if fine_tune:
        flags.append('--fine_tune')
    effective_labels_csv = labels_csv or (eval_dir(method, vit_size) / 'labels.csv')
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--classify',
        '--labels_csv', str(effective_labels_csv),
        '--checkpoint', str(checkpoint_path(method, vit_size)),
        '--framework', 'timm',
        '--method', get_method_for_extraction(method),
        '--vit_size', vit_size,
        '--output_dir', str(eval_dir(method, vit_size)),
    ] + flags
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## Extract Embeddings for All 9 Pilot Runs


Run this after the 9 training cells finish. For `mae_dino`, embedding extraction uses the stage-2 DINO encoder.

In [ ]:
import subprocess
import sys

for method, vit_size in EXPERIMENTS:
    try:
        run_extract_embeddings(method, vit_size, max_samples=10000)
    except subprocess.CalledProcessError as e:
        print(f"\n--- FAILED ON {method} {vit_size} ---")
        print("Re-running to capture exact error...")
        method_for_extract = get_method_for_extraction(method)
        output_path = embeddings_path(method, vit_size)
        cmd = [
            sys.executable,
            'scripts/extract_embeddings.py',
            '--checkpoint', str(checkpoint_path(method, vit_size)),
            '--framework', 'timm',
            '--method', method_for_extract,
            '--vit_size', vit_size,
            '--catalog_dir', str(CATALOG_DIR),
            '--max_samples', '10000',
            '--output', str(output_path),
        ]
        res = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
        print("STDERR:")
        print(res.stderr)
        break

## Optional: Cluster All 9 Embedding Sets

Run this if you want silhouette-backed clustering outputs and UMAP plots for every trained model before you choose a reference experiment for labeling.

In [ ]:
for method, vit_size in EXPERIMENTS:
    run_cluster_eval(method, vit_size, n_clusters=10)

## Cluster, Label, and Classify a Reference Pilot Run

Pick one trained experiment and use it to create a human-labeled `labels.csv` on the shared 10k corpus. Once that file exists, you can reuse it to run linear probes and fine-tuning for all 9 trained models.

Start with the best-looking candidate from the 9 runs, not all 9 at once.

In [ ]:
SELECTED_METHOD = 'mae'
SELECTED_SIZE = 'tiny'

print('Selected experiment:', SELECTED_METHOD, SELECTED_SIZE)
print('Checkpoint:', checkpoint_path(SELECTED_METHOD, SELECTED_SIZE))
print('Embeddings:', embeddings_path(SELECTED_METHOD, SELECTED_SIZE))
print('Eval dir:', eval_dir(SELECTED_METHOD, SELECTED_SIZE))

In [ ]:
run_cluster_eval(SELECTED_METHOD, SELECTED_SIZE, n_clusters=10)

In [ ]:
run_labeling(SELECTED_METHOD, SELECTED_SIZE, reps_per_cluster=20)

REFERENCE_LABELS_CSV = eval_dir(SELECTED_METHOD, SELECTED_SIZE) / 'labels.csv'
print('Reference labels:', REFERENCE_LABELS_CSV)

In [ ]:
import csv
from collections import Counter


def sanitize_labels_csv(labels_csv: Path) -> Path:
    sanitized_csv = labels_csv.with_name(f'{labels_csv.stem}_sanitized{labels_csv.suffix}')
    valid_labels = {'spiral', 'not_spiral'}
    cleaned_rows = []
    seen_rows = set()
    label_counts = Counter()
    removed_header_rows = 0
    removed_duplicate_rows = 0
    removed_invalid_rows = []

    with labels_csv.open(newline='') as handle:
        reader = csv.DictReader(handle)
        expected_fieldnames = ['file_path', 'cluster_id', 'label']
        if reader.fieldnames != expected_fieldnames:
            raise ValueError(f'Unexpected label CSV columns: {reader.fieldnames}')

        for row in reader:
            file_path = (row.get('file_path') or '').strip()
            cluster_id = (row.get('cluster_id') or '').strip()
            label = (row.get('label') or '').strip()

            if (file_path, cluster_id, label) == ('file_path', 'cluster_id', 'label'):
                removed_header_rows += 1
                continue

            if label not in valid_labels:
                removed_invalid_rows.append(row)
                continue

            row_key = (file_path, cluster_id, label)
            if row_key in seen_rows:
                removed_duplicate_rows += 1
                continue

            seen_rows.add(row_key)
            cleaned_rows.append({
                'file_path': file_path,
                'cluster_id': cluster_id,
                'label': label,
            })
            label_counts[label] += 1

    if not cleaned_rows:
        raise ValueError(f'No valid labels remained after sanitizing {labels_csv}')

    with sanitized_csv.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['file_path', 'cluster_id', 'label'])
        writer.writeheader()
        writer.writerows(cleaned_rows)

    print('Original labels CSV:', labels_csv)
    print('Sanitized labels CSV:', sanitized_csv)
    print('Removed duplicated header rows:', removed_header_rows)
    print('Removed exact duplicate rows:', removed_duplicate_rows)
    print('Removed invalid rows:', len(removed_invalid_rows))
    print('Label counts:', dict(label_counts))

    if set(label_counts) != valid_labels:
        raise ValueError(f'Expected binary labels {valid_labels}, found {set(label_counts)}')

    return sanitized_csv


REFERENCE_LABELS_CSV = sanitize_labels_csv(REFERENCE_LABELS_CSV)
print('Using sanitized labels for downstream classification:', REFERENCE_LABELS_CSV)

In [ ]:
run_classification(SELECTED_METHOD, SELECTED_SIZE, fine_tune=False)

In [ ]:
run_classification(SELECTED_METHOD, SELECTED_SIZE, fine_tune=True)

## Fine-tune `mae_dino` Only

Reuse `REFERENCE_LABELS_CSV` from the reference labeling run, then fine-tune only the three `mae_dino` checkpoints on Colab (`tiny`, `small`, `base`).

This keeps the run focused on the model family you actually want to compare, instead of sweeping every method/size pair.

In [ ]:
import csv
import importlib
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path


colab_drive = None
try:
    colab_drive = importlib.import_module('google.colab.drive')
except ModuleNotFoundError:
    pass

if colab_drive is not None:
    colab_drive.mount('/content/drive', force_remount=False)

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/vision-transformer')
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'output' / 'experiments'
REPO_ROOT = Path('/content/JWST-vision-transformer')

if not DRIVE_PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Missing Drive project root: {DRIVE_PROJECT_ROOT}')

if not REPO_ROOT.exists():
    subprocess.run([
        'git',
        'clone',
        'https://github.com/nntran15/JWST-vision-transformer.git',
        str(REPO_ROOT),
    ], check=True)

requirements_file = REPO_ROOT / 'requirements' / 'pytorch.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], check=True)

def locate_labels_csv() -> Path:
    candidates = [
        DRIVE_PROJECT_ROOT / 'output' / 'manual_labels' / 'spiral_vs_not_spiral.csv',
        REPO_ROOT / 'output' / 'manual_labels' / 'spiral_vs_not_spiral.csv',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    search_roots = [DRIVE_PROJECT_ROOT / 'output', REPO_ROOT / 'output']
    for root in search_roots:
        if not root.exists():
            continue
        matches = sorted(root.rglob('spiral_vs_not_spiral.csv'))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        'Could not find spiral_vs_not_spiral.csv under Drive output/ or the repo output/.'
    )

def sanitize_labels_csv(labels_csv: Path) -> Path:
    sanitized_csv = labels_csv.with_name(f'{labels_csv.stem}_sanitized{labels_csv.suffix}')
    valid_labels = {'spiral', 'not_spiral'}
    cleaned_rows = []
    seen_rows = set()
    label_counts = Counter()
    removed_header_rows = 0
    removed_duplicate_rows = 0
    removed_invalid_rows = []

    with labels_csv.open(newline='') as handle:
        reader = csv.DictReader(handle)
        expected_fieldnames = ['file_path', 'cluster_id', 'label']
        if reader.fieldnames != expected_fieldnames:
            raise ValueError(f'Unexpected label CSV columns: {reader.fieldnames}')

        for row in reader:
            file_path = (row.get('file_path') or '').strip()
            cluster_id = (row.get('cluster_id') or '').strip()
            label = (row.get('label') or '').strip()

            if (file_path, cluster_id, label) == ('file_path', 'cluster_id', 'label'):
                removed_header_rows += 1
                continue

            if label not in valid_labels:
                removed_invalid_rows.append(row)
                continue

            row_key = (file_path, cluster_id, label)
            if row_key in seen_rows:
                removed_duplicate_rows += 1
                continue

            seen_rows.add(row_key)
            cleaned_rows.append({
                'file_path': file_path,
                'cluster_id': cluster_id,
                'label': label,
            })
            label_counts[label] += 1

    if not cleaned_rows:
        raise ValueError(f'No valid labels remained after sanitizing {labels_csv}')

    with sanitized_csv.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['file_path', 'cluster_id', 'label'])
        writer.writeheader()
        writer.writerows(cleaned_rows)

    print('Original labels CSV:', labels_csv)
    print('Sanitized labels CSV:', sanitized_csv)
    print('Removed duplicated header rows:', removed_header_rows)
    print('Removed exact duplicate rows:', removed_duplicate_rows)
    print('Removed invalid rows:', len(removed_invalid_rows))
    print('Label counts:', dict(label_counts))

    if set(label_counts) != valid_labels:
        raise ValueError(f'Expected binary labels {valid_labels}, found {set(label_counts)}')

    return sanitized_csv

def checkpoint_path(method: str, vit_size: str) -> Path:
    ckpt_dir = OUTPUT_ROOT / f'pilot_{method}_timm_{vit_size}' / 'checkpoints'
    best_pt = ckpt_dir / 'checkpoint_best.pt'
    latest_pt = ckpt_dir / 'checkpoint_latest.pt'
    if best_pt.exists():
        return best_pt
    if latest_pt.exists():
        return latest_pt
    raise FileNotFoundError(f'Missing checkpoint for {method} {vit_size} under {ckpt_dir}')

def fine_tune_eval_dir(method: str, vit_size: str) -> Path:
    return OUTPUT_ROOT / f'pilot_{method}_timm_{vit_size}' / 'eval_manual_spiral'

def run_fine_tune_rerun(method: str, vit_size: str, labels_csv: Path) -> None:
    output_dir = fine_tune_eval_dir(method, vit_size)
    fine_tune_dir = output_dir / 'fine_tune'
    if fine_tune_dir.exists():
        shutil.rmtree(fine_tune_dir)
        print('Removed stale fine-tune directory:', fine_tune_dir)

    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--classify',
        '--fine_tune',
        '--labels_csv', str(labels_csv),
        '--checkpoint', str(checkpoint_path(method, vit_size)),
        '--framework', 'timm',
        '--method', method,
        '--vit_size', vit_size,
        '--output_dir', str(output_dir),
    ]
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

    report_path = fine_tune_dir / 'classification_report.txt'
    if not report_path.exists():
        raise FileNotFoundError(f'Expected fine-tune report was not written: {report_path}')
    print('Wrote:', report_path)

RERUN_FINE_TUNE_EXPERIMENTS = [
    ('dino', 'tiny'),
    ('dino', 'small'),
    ('dino', 'base'),
    ('mae', 'tiny'),
    ('mae', 'small'),
    ('mae', 'base'),
]

REFERENCE_LABELS_CSV = sanitize_labels_csv(locate_labels_csv())
print('Using sanitized labels for reruns:', REFERENCE_LABELS_CSV)

for method, vit_size in RERUN_FINE_TUNE_EXPERIMENTS:
    run_fine_tune_rerun(method, vit_size, REFERENCE_LABELS_CSV)

print('\nCompleted fine-tune reruns for the six inconsistent experiments.')

In [ ]:
for method, vit_size in EXPERIMENTS:
    run_classification(
        method,
        vit_size,
        fine_tune=False,
        labels_csv=REFERENCE_LABELS_CSV,
    )

In [ ]:
for method, vit_size in EXPERIMENTS:
    run_classification(
        method,
        vit_size,
        fine_tune=True,
        labels_csv=REFERENCE_LABELS_CSV,
    )

In [ ]:
MAE_DINO_EXPERIMENTS = [
    ('mae_dino', 'tiny'),
    ('mae_dino', 'small'),
    ('mae_dino', 'base'),
]

for method, vit_size in MAE_DINO_EXPERIMENTS:
    run_classification(
        method,
        vit_size,
        fine_tune=True,
        linear_probe=False,
        labels_csv=REFERENCE_LABELS_CSV,
    )